## ⛏️ Phase 1 : Backward Taint Analysis for Data Flow Pairs Extraction

In [1]:
# Imports
from   dotenv   import load_dotenv
import pandas   as pd
import datetime
import json
import sys
import os

# Add the upper folder to sys.path
sys.path.insert(0, "../")
from   App         import App

#### Parameters

In [2]:
# TMP Folder
TMP_PATH = "../../../0_Data/TMP/"

#### Initialization

In [3]:
print("⚡ Start - {} ⚡\n".format(datetime.datetime.now()))
startTime = datetime.datetime.now()

⚡ Start - 2024-08-02 13:21:47.726524 ⚡



In [4]:
# Create TMP Folder
if not os.path.exists(TMP_PATH):
    os.makedirs(TMP_PATH)
    print("📁🆕 Folder created       :", TMP_PATH)
else:
    print("📁✅ Folder already exists:", TMP_PATH)

📁✅ Folder already exists: ../../../0_Data/TMP/


#### 📥 1) Read Data

In [5]:
# Apps to be analyzed
INPUT_PATH   = "../../../0_Data/0_AndroCatSet.csv"
#INPUT_PATH   = "../../../0_Data/1_AndroCatSet_Mini.csv"

# Read the CSV File
appsDF = pd.read_csv(INPUT_PATH)

appsDF = appsDF.head(3)
appsDF

,sha256,pkgName,classID,googlePlayCategoryID,googlePlayDescription
0,9B30837BD2474AC3623A43D052F7ADC4C63E4AA9981F0F...,my.android.calc,Calculator,TOOLS,Handiness universal percentage calculator for ...
1,686DE8D8A0D08992CB135BC7A0500D0109D9697A1140B8...,com.vpn.basiccalculator,Calculator,TOOLS,CITIZEN CALCULATOR by ANGEL NX is best Mobile ...
2,A49864DCC90F6730569455BDFA39B4B7CF70AE0C34D656...,com.ba.fractioncalculator,Calculator,EDUCATION,"<b>Free offline fraction calculator</b> ✌, sup..."


### ⛏️ 2) Extract 

In [6]:
# Path to Android Platforms
load_dotenv()
ANDROID_PATH = os.getenv("ANDROID_PATH")

In [7]:
# Path to the Java Script used for Data Flows Extaction
JAVA_EXTRACTOR_PATH = "./damflow_extractor-1.0-jar-with-dependencies.jar"

# Direction
#DIRECTION		= "backward"
DIRECTION		= "forward"

# List of source and sinks
#SOURCES_APPROACH = "nosources"
SOURCES_APPROACH = "docflow"

# Timeout for Data Flow Analysis
TIMEOUT = 10

In [8]:
jsonAppArray         = []

def processRow(row):
    # Print message 
    print("--- 🔑 Analyzing APK: {} 🔑 ---\n".format(row['sha256']))

    # Create App instance
    app = App(row['sha256'], row['pkgName'], row['classID'])

    # Extract data flows
    try:
        app.extractDataFlows(TMP_PATH, JAVA_EXTRACTOR_PATH, ANDROID_PATH, DIRECTION, SOURCES_APPROACH, TIMEOUT)
    except Exception as e:
        print("\n--- ❌ Failed with Exception {}\n".format(e), flush=True)
        
    # Convert to JSON and append to list
    jsonAppString = app.toJsonString()
    jsonAppArray.append(json.loads(jsonAppString))
    

# Apply the function to each row in the DataFrame
_ = appsDF.apply(processRow, axis=1)

--- 🔑 Analyzing APK: 9B30837BD2474AC3623A43D052F7ADC4C63E4AA9981F0F9DAA658F825EBDB224 🔑 ---

--- 📥 Downloading APK
--- 📤 APK file with SHA256 already exists.
--- 💻 Executing: java -Xmx24g -Xss1g -jar ./damflow_extractor-1.0-jar-with-dependencies.jar -a ../../../0_Data/TMP/9B30837BD2474AC3623A43D052F7ADC4C63E4AA9981F0F9DAA658F825EBDB224.apk -p /home/marco/android/platforms -d forward -s docflow 
--- ⏲️ Timeout: 10 s



--- ⚠️ Timeout Reached.
--- 🗑️ Delete APK

--- ❌ Failed with Exception Timeout reached while executing the command.

--- 🔑 Analyzing APK: 686DE8D8A0D08992CB135BC7A0500D0109D9697A1140B8EDADF19B3B3BD2220C 🔑 ---

--- 📥 Downloading APK
--- 📤 APK file with SHA256 already exists.
--- 💻 Executing: java -Xmx24g -Xss1g -jar ./damflow_extractor-1.0-jar-with-dependencies.jar -a ../../../0_Data/TMP/686DE8D8A0D08992CB135BC7A0500D0109D9697A1140B8EDADF19B3B3BD2220C.apk -p /home/marco/android/platforms -d forward -s docflow 
--- ⏲️ Timeout: 10 s

--- ⚠️ Timeout Reached.
--- 🗑️ Delete APK

--- ❌ Failed with Exception Timeout reached while executing the command.

--- 🔑 Analyzing APK: A49864DCC90F6730569455BDFA39B4B7CF70AE0C34D656D078790CBA3521C0A0 🔑 ---

--- 📥 Downloading APK
--- 📤 APK file with SHA256 already exists.
--- 💻 Executing: java -Xmx24g -Xss1g -jar ./damflow_extractor-1.0-jar-with-dependencies.jar -a ../../../0_Data/TMP/A49864DCC90F6730569455BDFA39B4B7CF70AE0C34D656D078790CBA3521C0A0.apk -p /

##### 🔚 End

In [9]:
endTime = datetime.datetime.now()
print("🔚 --- End - {} --- 🔚".format(endTime))

# Assuming endTime and startTime are in seconds
totalTime = endTime - startTime
minutes = totalTime.total_seconds() // 60
seconds = totalTime.total_seconds() % 60
print("⏱️ --- Time: {:02d} minutes and {:02d} seconds --- ⏱️".format(int(minutes), int(seconds)))

🔚 --- End - 2024-08-02 13:22:17.947445 --- 🔚
⏱️ --- Time: 00 minutes and 30 seconds --- ⏱️
